# Per-lineage effect sizes: delta_psi (NMI, delta_psi_max) and rho_W

Computes the model-derived effect size delta_psi (and its companions delta_psi_max, NMI) and the model-independent gene-level concordance metric rho_W for each lineage, restricted to the controls cohort (induced=False, N ~ 6,000 cells per lineage; same population used by the DirMult LRT).

Same methods as for the mouse data: `analysis/mouse_svz/01_260309-calculate_delta_psi.py` (delta_psi) and `analysis/mouse_svz/05_260504-genelevel-W_spearman_graphsmoothed.ipynb` (rho_W).

Pipeline per lineage:

1. Subset the raw and harmony h5ads to the cells listed in `cells_used.csv` for that lineage. Save both as standalone h5ads (reused by downstream notebooks).
2. Run `08_260504-calculate_delta_psi.py` on the raw subset via subprocess, with `--pseudotime_col dpt_pseudotime_<lineage>`, `--batch_col batch_number`, `--gene_col name`. Output: per-gene and per-site delta_psi CSVs.
3. Recompute Harmony-based neighbors graph on the harmony subset (the original graph was on all 71,861 cells). Apply 2 rounds of row-normalised connectivities to the raw counts, compute per-cell W = sum_k(r_k \* pi_k) using normalised utr_size ranks, and Spearman rho_W against pseudotime per gene over expressing cells (>=30, gene_total>0).
4. Convert the 2-round smoothed counts to within-gene proportions and save as `smoothed_proportions.h5ad` for downstream visualisation notebooks (W(t) trajectories, stacked area plots).

Outputs under `$M26_DATA_ROOT/3utr/organoids/knoblich/utrcalling/effect_sizes/`:

- `<lineage>_controls_raw_pt.h5ad`, `<lineage>_controls_harmony_pt.h5ad` (subsets)
- `<lineage>/delta_psi_per_gene_df5_q0.1-0.9.csv`, `<lineage>/delta_psi_per_site_df5_q0.1-0.9.csv`
- `<lineage>/rho_W_per_gene.csv`
- `<lineage>/smoothed_proportions.h5ad`


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
import sys
import json
import time
import subprocess
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import matplotlib.pyplot as plt

from scipy.sparse import issparse, diags
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

LAYOUT_DPI = 300
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": LAYOUT_DPI,
        "font.size": 7,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": 7,
        "axes.labelsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)
sc.settings.verbosity = 1
print("Setup complete.")

In [ ]:
LINEAGE_TAG = {"Inhibitory neurons": "inhibitory", "Excitatory neurons": "excitatory"}
RANDOM_STATE = 42
N_NEIGHBORS = 10
N_SMOOTH_ROUNDS = 2
SPLINE_DF = 5
Q_EARLY = 0.1
Q_LATE = 0.9
N_GRID = 50
MIN_COUNTS = 100
MIN_CELLS_PER_GENE = 30
NCORES = 8

BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_DIR = BASEPATH / "utrcalling"
DM_RESULTS_DIR = UTR_DIR / "dirmult_results"
EFFECT_SIZES_DIR = UTR_DIR / "effect_sizes"
EFFECT_SIZES_DIR.mkdir(parents=True, exist_ok=True)

RAW_PT_PATH = UTR_DIR / "merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"
HARMONY_PT_PATH = (
    UTR_DIR / "merged_bw12_above10cts_apa_named_percentages_annotated_harmony_pt.h5ad"
)

DELTA_PSI_SCRIPT = Path(
    f"{REPO_ROOT}/analysis/organoids/pseudotime/08_260504-calculate_delta_psi.py"
)
PYTHON_BIN = sys.executable

assert RAW_PT_PATH.exists(), f"Missing raw h5ad: {RAW_PT_PATH}"
assert HARMONY_PT_PATH.exists(), f"Missing harmony h5ad: {HARMONY_PT_PATH}"
assert DELTA_PSI_SCRIPT.exists(), f"Missing delta_psi script: {DELTA_PSI_SCRIPT}"

print(f"BASEPATH:        {BASEPATH}")
print(f"raw_pt:          {RAW_PT_PATH}")
print(f"harmony_pt:      {HARMONY_PT_PATH}")
print(f"effect_sizes:    {EFFECT_SIZES_DIR}")
print(f"delta_psi script: {DELTA_PSI_SCRIPT}")

## Load full h5ad inputs

Load both the raw and the harmony h5ads in full. They will be subsetted in-memory per lineage.


In [ ]:
t0 = time.time()
print("Loading raw_pt h5ad ...")
adata_raw_full = sc.read_h5ad(RAW_PT_PATH)
print(f"  shape: {adata_raw_full.shape} ({time.time()-t0:.1f}s)")

t1 = time.time()
print("Loading harmony_pt h5ad ...")
adata_harm_full = sc.read_h5ad(HARMONY_PT_PATH)
print(f"  shape: {adata_harm_full.shape} ({time.time()-t1:.1f}s)")
print(f"  obsm: {list(adata_harm_full.obsm.keys())}")

assert (
    adata_raw_full.obs_names == adata_harm_full.obs_names
).all(), "raw and harmony h5ads must have identical obs_names ordering"
print(f"raw n_vars: {adata_raw_full.n_vars}, harmony n_vars: {adata_harm_full.n_vars}")

## Per-lineage processing

For each lineage:

1. Build the controls subset (from cells_used.csv) of both raw and harmony h5ads.
2. Run the delta_psi script on the raw subset.
3. Compute rho_W on the harmony subset (recomputing neighbors + graph smoothing the raw counts).
4. Save the smoothed proportions h5ad.


In [ ]:
def make_controls_subset(adata_full, cells_csv_path, lineage_tag, kind):
    # Subset adata_full to the cells listed in cells_used.csv. Save to disk.
    cells_df = pd.read_csv(cells_csv_path)
    barcodes = cells_df["cell_barcode"].astype(str).values
    out_path = EFFECT_SIZES_DIR / f"{lineage_tag}_controls_{kind}_pt.h5ad"
    if out_path.exists():
        print(f"  [{lineage_tag} {kind}] already exists: {out_path}")
        return out_path
    missing = set(barcodes) - set(adata_full.obs_names.astype(str))
    if missing:
        raise RuntimeError(
            f"{len(missing)} barcodes from cells_used.csv not found in {kind} h5ad"
        )
    a = adata_full[barcodes].copy()
    for col in [
        "celltype",
        "lineage",
        "induced",
        "batch_number",
        "sample_id",
        "super_batch",
    ]:
        if col in a.obs.columns:
            a.obs[col] = a.obs[col].astype(str).astype("category")
    if a.X.dtype != np.float64:
        a.X = a.X.astype(np.float64)
    a.write_h5ad(out_path)
    print(f"  [{lineage_tag} {kind}] saved {a.shape} -> {out_path}")
    return out_path

In [ ]:
def compute_rho_W(
    adata_harm_subset,
    adata_raw_subset,
    n_rounds=N_SMOOTH_ROUNDS,
    n_neighbors=N_NEIGHBORS,
    random_state=RANDOM_STATE,
    min_cells=MIN_CELLS_PER_GENE,
    gene_col="name",
):
    # Recompute neighbors on subset, graph-smooth raw counts, compute per-gene rho_W.
    assert (
        adata_harm_subset.obs_names == adata_raw_subset.obs_names
    ).all(), "harmony and raw subsets must have identical obs_names"

    print("  recomputing neighbors on the harmony subset ...")
    sc.pp.neighbors(
        adata_harm_subset,
        n_neighbors=n_neighbors,
        use_rep="X_pca_harmony",
        random_state=random_state,
        method="umap",
    )

    pt_cols = [
        c for c in adata_harm_subset.obs.columns if c.startswith("dpt_pseudotime")
    ]
    pt_col = next(c for c in pt_cols if not adata_harm_subset.obs[c].isna().any())
    pseudotime = adata_harm_subset.obs[pt_col].values.astype(float)
    print(
        f"  pseudotime column: {pt_col} (range {pseudotime.min():.3f} - {pseudotime.max():.3f})"
    )

    X_raw = adata_raw_subset.X
    if issparse(X_raw):
        X_raw = X_raw.toarray().astype(np.float64)
    else:
        X_raw = np.asarray(X_raw, dtype=np.float64)
    print(f"  raw X: {X_raw.shape}, sparsity {(X_raw == 0).mean():.1%}")

    conn = adata_harm_subset.obsp["connectivities"].copy()
    rs = np.array(conn.sum(axis=1)).flatten()
    rs[rs == 0] = 1
    norm_conn = diags(1.0 / rs) @ conn

    X_smooth = X_raw.copy()
    for i in range(n_rounds):
        X_smooth = norm_conn @ X_smooth
        print(f"  smoothing round {i+1}/{n_rounds}")
    print(f"  smoothed sparsity: {(X_smooth == 0).mean():.1%}")

    var = adata_raw_subset.var
    var_names = list(adata_raw_subset.var_names)
    var_to_idx = {v: i for i, v in enumerate(var_names)}

    gene_to_sites = defaultdict(list)
    for site_name, row in var.iterrows():
        gid = row[gene_col]
        if pd.isna(gid):
            continue
        gene_to_sites[gid].append(site_name)
    multi = {g: s for g, s in gene_to_sites.items() if len(s) >= 2}
    gene_id_to_name = (
        var.groupby(gene_col, observed=True)["gene_name"].first().to_dict()
    )
    print(f"  multi-site genes: {len(multi)} / {len(gene_to_sites)}")

    all_utr_sizes = var["utr_size"].values.astype(float)
    results = []
    for i, (gid, sites) in enumerate(multi.items()):
        if (i + 1) % 1000 == 0:
            print(f"    {i+1}/{len(multi)}")
        site_idxs = np.fromiter(
            (var_to_idx[s] for s in sites), dtype=np.int64, count=len(sites)
        )
        K = len(site_idxs)
        raw_ranks = rankdata(all_utr_sizes[site_idxs], method="average")
        ranks = (raw_ranks - 1) / (K - 1)

        sub = X_smooth[:, site_idxs]
        gene_total = sub.sum(axis=1)
        expressing = gene_total > 0
        n_expr = int(expressing.sum())
        if n_expr < min_cells:
            results.append(
                (gid, gene_id_to_name.get(gid, ""), K, n_expr, np.nan, np.nan)
            )
            continue
        weighted = sub[expressing] @ ranks
        W_cell = weighted / gene_total[expressing]
        if W_cell.std() == 0:
            results.append((gid, gene_id_to_name.get(gid, ""), K, n_expr, 0.0, 1.0))
            continue
        rho, pval = spearmanr(pseudotime[expressing], W_cell)
        results.append(
            (gid, gene_id_to_name.get(gid, ""), K, n_expr, float(rho), float(pval))
        )

    df = pd.DataFrame(
        results,
        columns=["gene_id", "gene_name", "n_sites", "n_cells", "rho_W", "pvalue"],
    )
    valid = df["pvalue"].notna()
    padj = np.full(len(df), np.nan)
    if valid.any():
        _, padj_v, _, _ = multipletests(df.loc[valid, "pvalue"].values, method="fdr_bh")
        padj[valid.values] = padj_v
    df["padj_BH"] = padj
    df["abs_rho_W"] = df["rho_W"].abs()
    df = df.sort_values("abs_rho_W", ascending=False, na_position="last").reset_index(
        drop=True
    )
    return df, X_smooth, pseudotime

In [ ]:
def build_smoothed_proportions_adata(adata_template, X_smooth, gene_col="name"):
    # Build an AnnData of within-gene proportions from smoothed counts.
    var = adata_template.var
    var_names = list(adata_template.var_names)
    var_to_idx = {v: i for i, v in enumerate(var_names)}
    gene_to_sites = defaultdict(list)
    for site_name, row in var.iterrows():
        gid = row[gene_col]
        if pd.isna(gid):
            continue
        gene_to_sites[gid].append(site_name)
    multi = {g: s for g, s in gene_to_sites.items() if len(s) >= 2}

    P_smooth = np.zeros_like(X_smooth)
    for gid, sites in multi.items():
        site_idxs = [var_to_idx[s] for s in sites]
        gene_total = X_smooth[:, site_idxs].sum(axis=1)
        zero_mask = gene_total == 0
        K = len(site_idxs)
        for j, idx in enumerate(site_idxs):
            P_smooth[:, idx] = np.where(
                zero_mask,
                1.0 / K,
                X_smooth[:, idx] / np.where(zero_mask, 1.0, gene_total),
            )

    out = anndata.AnnData(
        X=P_smooth, obs=adata_template.obs.copy(), var=adata_template.var.copy()
    )
    out.uns["smoothing_rounds"] = N_SMOOTH_ROUNDS
    out.uns["smoothing_n_neighbors"] = N_NEIGHBORS
    out.uns["smoothing_random_state"] = RANDOM_STATE
    return out, multi

In [ ]:
def run_delta_psi(raw_subset_path, lineage_tag, output_subdir):
    # Invoke the standalone 08_260504-calculate_delta_psi.py script.
    output_subdir.mkdir(parents=True, exist_ok=True)
    pt_col = f"dpt_pseudotime_{lineage_tag}"
    cmd = [
        PYTHON_BIN,
        str(DELTA_PSI_SCRIPT),
        str(raw_subset_path),
        "--pseudotime_col",
        pt_col,
        "--batch_col",
        "batch_number",
        "--gene_col",
        "name",
        "--output_dir",
        str(output_subdir),
        "--spline_df",
        str(SPLINE_DF),
        "--q_early",
        str(Q_EARLY),
        "--q_late",
        str(Q_LATE),
        "--n_grid",
        str(N_GRID),
        "--min_counts",
        str(MIN_COUNTS),
        "--min_cells",
        str(MIN_CELLS_PER_GENE),
        "--n_cores",
        str(NCORES),
    ]
    print(f"  invoking: {' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print("STDERR tail:")
        print(proc.stderr[-2000:])
        raise RuntimeError(
            f"delta_psi script failed for {lineage_tag} (rc={proc.returncode})"
        )
    tag = f"df{SPLINE_DF}_q{Q_EARLY}-{Q_LATE}"
    gene_csv = output_subdir / f"delta_psi_per_gene_{tag}.csv"
    site_csv = output_subdir / f"delta_psi_per_site_{tag}.csv"
    assert gene_csv.exists() and site_csv.exists(), "delta_psi output CSVs missing"
    return gene_csv, site_csv

In [ ]:
def process_lineage(lineage_name, lineage_tag):
    print(f"\n{'=' * 60}")
    print(f"=== {lineage_name} ({lineage_tag}) ===")
    print(f"{'=' * 60}")

    cells_csv = (
        DM_RESULTS_DIR / lineage_tag / "controls_only_batch_number" / "cells_used.csv"
    )
    assert cells_csv.exists(), f"Missing {cells_csv}"

    out_subdir = EFFECT_SIZES_DIR / lineage_tag
    out_subdir.mkdir(parents=True, exist_ok=True)

    print(f"\n[step 1] Build controls subsets")
    raw_subset_path = make_controls_subset(
        adata_raw_full, cells_csv, lineage_tag, "raw"
    )
    harm_subset_path = make_controls_subset(
        adata_harm_full, cells_csv, lineage_tag, "harmony"
    )

    print(f"\n[step 2] Run delta_psi")
    gene_csv, site_csv = run_delta_psi(raw_subset_path, lineage_tag, out_subdir)
    print(f"  gene CSV: {gene_csv}")
    print(f"  site CSV: {site_csv}")

    print(f"\n[step 3] Compute rho_W per gene")
    a_harm = sc.read_h5ad(harm_subset_path)
    a_raw = sc.read_h5ad(raw_subset_path)
    df_rho, X_smooth, pseudotime = compute_rho_W(a_harm, a_raw, gene_col="name")
    rho_W_csv = out_subdir / "rho_W_per_gene.csv"
    df_rho.to_csv(rho_W_csv, index=False)
    print(f"  saved {df_rho.shape} -> {rho_W_csv}")
    print(f"  |rho_W| > 0.3: {(df_rho['abs_rho_W'] > 0.3).sum()}")

    print(f"\n[step 4] Save smoothed proportions h5ad")
    smoothed_adata, _ = build_smoothed_proportions_adata(
        a_raw, X_smooth, gene_col="name"
    )
    smoothed_path = out_subdir / "smoothed_proportions.h5ad"
    smoothed_adata.write_h5ad(smoothed_path)
    print(f"  saved {smoothed_adata.shape} -> {smoothed_path}")

    return {
        "lineage": lineage_name,
        "tag": lineage_tag,
        "raw_subset": str(raw_subset_path),
        "harmony_subset": str(harm_subset_path),
        "delta_psi_per_gene": str(gene_csv),
        "delta_psi_per_site": str(site_csv),
        "rho_W_per_gene": str(rho_W_csv),
        "smoothed_proportions": str(smoothed_path),
        "n_genes_rho_W": len(df_rho),
        "n_sig_rho_W_03": int((df_rho["abs_rho_W"] > 0.3).sum()),
    }


lineage_summaries = {}
for lname, ltag in LINEAGE_TAG.items():
    lineage_summaries[ltag] = process_lineage(lname, ltag)

print("\n=== ALL LINEAGES DONE ===")
for ltag, summary in lineage_summaries.items():
    print(f"\n[{ltag}]")
    for k, v in summary.items():
        print(f"  {k}: {v}")

## Sanity diagnostics

Quick distributional checks on the produced effect sizes.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for row, ltag in enumerate(LINEAGE_TAG.values()):
    s = lineage_summaries[ltag]
    dpsi = pd.read_csv(s["delta_psi_per_gene"])
    rho = pd.read_csv(s["rho_W_per_gene"])

    axes[row, 0].hist(
        dpsi["delta_psi"].dropna(),
        bins=60,
        color="#2980b9",
        edgecolor="black",
        linewidth=0.3,
    )
    axes[row, 0].axvline(0, color="black", linewidth=0.7, linestyle="--")
    axes[row, 0].set_title(f"{ltag}: delta_psi (n={dpsi['delta_psi'].notna().sum()})")
    axes[row, 0].set_xlabel("delta_psi")
    axes[row, 0].set_ylabel("genes")

    axes[row, 1].hist(
        dpsi["nmi"].dropna(), bins=40, color="#16a085", edgecolor="black", linewidth=0.3
    )
    axes[row, 1].set_title(f"{ltag}: NMI")
    axes[row, 1].set_xlabel("NMI")
    axes[row, 1].set_ylabel("genes")

    axes[row, 2].hist(
        rho["rho_W"].dropna(),
        bins=60,
        color="#c0392b",
        edgecolor="black",
        linewidth=0.3,
    )
    axes[row, 2].axvline(0, color="black", linewidth=0.7, linestyle="--")
    axes[row, 2].set_title(f"{ltag}: rho_W (n={rho['rho_W'].notna().sum()})")
    axes[row, 2].set_xlabel("rho_W")
    axes[row, 2].set_ylabel("genes")

    sig = rho["padj_BH"] < 0.05
    axes[row, 3].hist(
        rho.loc[~sig, "abs_rho_W"].dropna(),
        bins=40,
        alpha=0.5,
        label="padj_BH >= 0.05",
        color="lightgrey",
        edgecolor="black",
        linewidth=0.3,
    )
    axes[row, 3].hist(
        rho.loc[sig, "abs_rho_W"].dropna(),
        bins=40,
        alpha=0.7,
        label="padj_BH < 0.05",
        color="firebrick",
        edgecolor="black",
        linewidth=0.3,
    )
    axes[row, 3].set_title(f"{ltag}: |rho_W| stratified")
    axes[row, 3].set_xlabel("|rho_W|")
    axes[row, 3].set_ylabel("genes")
    axes[row, 3].legend(fontsize=6)

plt.tight_layout()
fig_path = EFFECT_SIZES_DIR / "diagnostics_distributions.pdf"
plt.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"saved {fig_path}")

In [ ]:
summary_path = EFFECT_SIZES_DIR / "lineage_summary.json"
with open(summary_path, "w") as f:
    json.dump(lineage_summaries, f, indent=2)
print(f"saved {summary_path}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling")
EFFECT_SIZES_DIR = BASE / "effect_sizes"
COMPOSITE_DIR = BASE / "composite_ranking"
LINEAGES = ["inhibitory", "excitatory"]


bins = np.linspace(0, 1, 31)
summary_rows = []
fig, axes = plt.subplots(1, len(LINEAGES), figsize=(9, 3.5), sharey=True)

for ax, lineage in zip(np.atleast_1d(axes), LINEAGES):
    dpsi = pd.read_csv(
        EFFECT_SIZES_DIR / lineage / "delta_psi_per_gene_df5_q0.1-0.9.csv"
    )
    comp = pd.read_csv(COMPOSITE_DIR / f"composite_ranking_optimized2_{lineage}.csv")
    sig_ids = set(comp.loc[comp["padj"] < 0.05, "gene_id"])
    dpsi = dpsi.copy()
    dpsi["is_sig_lrt"] = dpsi["gene_id"].isin(sig_ids)
    all_nmi = dpsi["nmi"].dropna()
    sig_nmi = dpsi.loc[dpsi["is_sig_lrt"], "nmi"].dropna()
    ax.hist(
        all_nmi,
        bins=bins,
        density=True,
        color="lightgray",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.8,
        label=f"all (n={len(all_nmi):,})",
    )
    ax.hist(
        sig_nmi,
        bins=bins,
        density=True,
        color="#16a085",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.65,
        label=f"LRT padj<0.05 (n={len(sig_nmi):,})",
    )

    ax.axvline(all_nmi.median(), color="0.35", linestyle="--", linewidth=1.0)
    ax.axvline(sig_nmi.median(), color="#0b5345", linestyle="--", linewidth=1.0)
    ax.set_title(lineage)
    ax.set_xlabel("NMI")
    ax.set_ylabel("density")
    ax.legend(fontsize=6, frameon=False)

    summary_rows.append(
        {
            "lineage": lineage,
            "n_all": len(all_nmi),
            "n_sig_lrt": len(sig_nmi),
            "median_nmi_all": all_nmi.median(),
            "median_nmi_sig_lrt": sig_nmi.median(),
            "frac_all_nmi_lt_0.1": (all_nmi < 0.1).mean(),
            "frac_sig_nmi_lt_0.1": (sig_nmi < 0.1).mean(),
            "frac_all_nmi_gt_0.5": (all_nmi > 0.5).mean(),
            "frac_sig_nmi_gt_0.5": (sig_nmi > 0.5).mean(),
        }
    )

plt.tight_layout()
plt.show()

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(3))